In [254]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report,r2_score,recall_score
from sklearn.neighbors import KNeighborsClassifier


In [26]:
df = pd.read_csv("novagen_dataset.csv")
df.head()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,False,True,True,False,False
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,False,False,True,False,False
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,False,True,False,True,False


# voting


In [52]:
le = LabelEncoder()

TO_MAKE_ENCODING=["Diet_Type__Vegan","Diet_Type__Vegetarian","Blood_Group_AB","Blood_Group_B","Blood_Group_O"]
for col in TO_MAKE_ENCODING:
    df[col] = le.fit_transform(df[col])


In [108]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

In [116]:
x=df.drop("Target",axis=1)
y=df["Target"]

In [118]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [190]:
lr=LogisticRegression()
svc=SVC()
dtc=DecisionTreeClassifier()

In [192]:
voting_clf=VotingClassifier(
    estimators=[
    ('lr',lr),
    ('svc',svc),
    ('dtc',dtc)
    ],
)


In [194]:
voting_clf.fit(x_train,y_train)

C:\Users\samiy\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


VotingClassifier(estimators=[('lr', LogisticRegression()), ('svc', SVC()),
                             ('dtc', DecisionTreeClassifier())])

In [198]:
y_pred = voting_clf.predict(x_test)
print("accuracy:", accuracy_score(y_pred, y_test))
print("classification report:", classification_report(y_pred, y_test))

accuracy: 0.8403141361256544
classification report:               precision    recall  f1-score   support

           0       0.89      0.80      0.84      1003
           1       0.80      0.89      0.84       907

    accuracy                           0.84      1910
   macro avg       0.84      0.84      0.84      1910
weighted avg       0.85      0.84      0.84      1910



# XGB CLASSIFIER

In [136]:
import xgboost as xgb

In [200]:
xgb_clf=xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    leaning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)
    

In [202]:
xgb_clf.fit(x_train, y_train)

C:\Users\samiy\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [15:35:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "leaning_rate" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              leaning_rate=0.1, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [204]:
y_pred = xgb_clf.predict(x_test)
print("accuracy:", accuracy_score(y_test, y_pred))
print("classification report:\n", classification_report(y_test, y_pred))

accuracy: 0.93717277486911
classification report:
               precision    recall  f1-score   support

           0       0.93      0.94      0.93       900
           1       0.94      0.94      0.94      1010

    accuracy                           0.94      1910
   macro avg       0.94      0.94      0.94      1910
weighted avg       0.94      0.94      0.94      1910



# stacking classifier

In [206]:
meta_model = LogisticRegression()
stk_clf = StackingClassifier(
    estimators=[
        ('lr',lr),
        ('svc',svc),
        ('dtc',dtc)
    ],
    final_estimator=meta_model,
    cv=5
)

In [208]:
stk_clf.fit(x_train,y_train)

C:\Users\samiy\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\samiy\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

StackingClassifier(cv=5,
                   estimators=[('lr', LogisticRegression()), ('svc', SVC()),
                               ('dtc', DecisionTreeClassifier())],
                   final_estimator=LogisticRegression())

In [214]:
y_pred=stk_clf.predict(x_test)
print("accuracy:", accuracy_score(y_pred, y_test))

accuracy: 0.8947643979057591


# graident bosting classifier

In [219]:
from sklearn.ensemble import GradientBoostingClassifier
gbc =GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth = 3,
    random_state =42,
    subsample=0.8
    
)
gbc.fit(x_train,y_train)

GradientBoostingClassifier(n_estimators=150, random_state=42, subsample=0.8)

In [226]:
y_pred = gbc.predict(x_test)
print("r2 score",r2_score(y_test, y_pred))
print("accuracy score",accuracy_score(y_test, y_pred))

r2 score 0.7016281628162817
accuracy score 0.9256544502617801


# voting classifier apna collage

In [262]:

df = pd.read_csv("novagen_dataset.csv")

# Split features and target
X = df.drop("Target", axis=1)
y = df["Target"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



voting_clf = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, solver="liblinear")),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
    ],
    voting="soft"
)

voting_clf.fit(X_train_scaled, y_train)

VotingClassifier(estimators=[('lr',
                              LogisticRegression(max_iter=1000,
                                                 solver='liblinear')),
                             ('knn', KNeighborsClassifier()),
                             ('rf',
                              RandomForestClassifier(n_estimators=200,
                                                     random_state=42))],
                 voting='soft')

In [264]:


y_pred_vote = voting_clf.predict(X_test_scaled)

print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred_vote))
print("Voting Classifier Recall:", recall_score(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote))


Voting Classifier Accuracy: 0.9157068062827225
Voting Classifier Recall: 0.929718875502008
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       914
           1       0.91      0.93      0.92       996

    accuracy                           0.92      1910
   macro avg       0.92      0.92      0.92      1910
weighted avg       0.92      0.92      0.92      1910



In [272]:
stck_clf = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, solver="liblinear")),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
    ],
    final_estimator=meta_model,
    cv=5

)
stck_clf.fit(X_train_scaled, y_train)

StackingClassifier(cv=5,
                   estimators=[('lr',
                                LogisticRegression(max_iter=1000,
                                                   solver='liblinear')),
                               ('knn', KNeighborsClassifier()),
                               ('rf',
                                RandomForestClassifier(n_estimators=200,
                                                       random_state=42))],
                   final_estimator=LogisticRegression())

In [277]:
y_pred=stk_clf.predict(x_test)
print("accuracy:", accuracy_score(y_pred, y_test))
print("classification report:\n", classification_report(y_test, y_pred))

accuracy: 0.5010471204188481
classification report:
               precision    recall  f1-score   support

           0       0.48      0.48      0.48       914
           1       0.52      0.52      0.52       996

    accuracy                           0.50      1910
   macro avg       0.50      0.50      0.50      1910
weighted avg       0.50      0.50      0.50      1910

